
# Text Classification: TF‑IDF + XGBoost vs Embeddings + LR vs DistilBERT (Colab)

This notebook compares three approaches on the **20 Newsgroups** dataset:

1. **TF‑IDF + XGBoost** (strong classical baseline)  
2. **Sentence‑Transformer embeddings + Logistic Regression** (semantic baseline)  
3. **DistilBERT fine‑tuning** (modern transformer fine-tune)

We measure: **Accuracy**, **Macro‑F1**, **training time**, and **inference time**. We also plot confusion matrices.

> Tip: In Colab, go to **Runtime → Change runtime type → GPU (T4/A100)** for the DistilBERT section.


## 0. Setup

In [ ]:

# If you're on Colab, uncomment the next cell to ensure required libraries are installed.
# You can safely re-run this cell.

!pip -q install xgboost scikit-learn datasets transformers accelerate evaluate sentence-transformers torch torchvision --upgrade


## 1. Imports & Utilities

In [ ]:

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils import Bunch

# Approach 1
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

# Approach 2
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

# Approach 3
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

import evaluate as hf_evaluate

RANDOM_STATE = 42

def plot_confusion_matrix(cm, class_names, title):
    fig = plt.figure(figsize=(8, 8))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=90)
    plt.yticks(tick_marks, class_names)
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


## 2. Load Dataset (20 Newsgroups)

In [ ]:

# We'll use a subset of the 20NG dataset for speed. You can set to None for all categories.
categories = [
    'alt.atheism',
    'comp.graphics',
    'comp.sys.mac.hardware',
    'rec.autos',
    'rec.sport.baseball',
    'sci.electronics',
    'sci.space',
    'talk.politics.mideast'
]

newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers','footers','quotes'))
X_text = newsgroups.data
y = newsgroups.target
target_names = list(newsgroups.target_names)

print(f"Total documents: {len(X_text)}, classes: {len(target_names)}")

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")


## 3. Baseline A — TF‑IDF + XGBoost

In [ ]:

# Vectorize with TF-IDF
tv = TfidfVectorizer(stop_words='english', max_features=100_000, ngram_range=(1,2), min_df=2, max_df=0.9)
t0 = time.time()
X_train_tfidf = tv.fit_transform(X_train_text)
X_test_tfidf  = tv.transform(X_test_text)
vec_time = time.time() - t0
print(f"TF-IDF shapes: train={X_train_tfidf.shape}, test={X_test_tfidf.shape} (built in {vec_time:.2f}s)")

# XGBoost multi-class
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=len(target_names),
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

t0 = time.time()
xgb.fit(X_train_tfidf, y_train)
train_time = time.time() - t0

t0 = time.time()
y_pred = xgb.predict(X_test_tfidf)
infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average='macro')

print(f"TF-IDF + XGBoost -> Acc: {acc:.4f}, Macro-F1: {f1m:.4f}, Train: {train_time:.2f}s, Inference: {infer_time:.2f}s")
cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(cm, target_names, "Confusion Matrix — TF-IDF + XGBoost")


## 4. Baseline B — Sentence‑Transformer Embeddings + Logistic Regression

In [ ]:

model_name = "sentence-transformers/all-MiniLM-L6-v2"
st_model = SentenceTransformer(model_name, device='cuda' if torch.cuda.is_available() else 'cpu')

t0 = time.time()
X_train_emb = st_model.encode(X_train_text, batch_size=64, convert_to_numpy=True, show_progress_bar=True)
X_test_emb  = st_model.encode(X_test_text, batch_size=64, convert_to_numpy=True, show_progress_bar=True)
emb_time = time.time() - t0
print(f"Built embeddings: train={X_train_emb.shape}, test={X_test_emb.shape} (in {emb_time:.2f}s)")

clf = LogisticRegression(max_iter=2000, n_jobs=-1, solver='saga', multi_class='multinomial', verbose=0)
t0 = time.time()
clf.fit(X_train_emb, y_train)
train_time = time.time() - t0

t0 = time.time()
y_pred = clf.predict(X_test_emb)
infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average='macro')

print(f"Embeddings + LR -> Acc: {acc:.4f}, Macro-F1: {f1m:.4f}, Train: {train_time:.2f}s, Inference: {infer_time:.2f}s")
cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(cm, target_names, "Confusion Matrix — Embeddings + LR")


## 5. DistilBERT Fine‑Tuning (Hugging Face Transformers)

In [ ]:

# Prepare a Hugging Face Dataset
train_df = pd.DataFrame({'text': X_train_text, 'label': y_train})
test_df  = pd.DataFrame({'text': X_test_text, 'label': y_test})
hf_train = Dataset.from_pandas(train_df)
hf_test  = Dataset.from_pandas(test_df)

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_train = hf_train.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized_test  = hf_test.map(tokenize_fn, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
id2label = {i: name for i, name in enumerate(target_names)}
label2id = {name: i for i, name in enumerate(target_names)}

accuracy = hf_evaluate.load("accuracy")
f1 = hf_evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(target_names),
    id2label=id2label,
    label2id=label2id
).to(device)

training_args = TrainingArguments(
    output_dir="./distilbert-20ng",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    load_best_model_at_end=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

t0 = time.time()
trainer.train()
train_time = time.time() - t0

eval_metrics = trainer.evaluate()
print(f"DistilBERT -> Acc: {eval_metrics['eval_accuracy']:.4f}, Macro-F1: {eval_metrics['eval_f1_macro']:.4f}, Train: {train_time:.2f}s")

# Predictions for confusion matrix
preds = trainer.predict(tokenized_test).predictions
y_pred = np.argmax(preds, axis=1)
cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(cm, target_names, "Confusion Matrix — DistilBERT")


## 6. (Optional) Consolidate Results Into a Table

In [ ]:

# Rerun sections above to collect metrics, or adapt to log metrics.
# Below is a skeleton to help collect them programmatically if you store them along the way.

# Example structure you can populate:
results = pd.DataFrame([
    # Fill with your actual numbers (copy from printed outputs)
    # {"approach": "TF-IDF + XGBoost", "accuracy": 0.00, "f1_macro": 0.00, "train_s": 0.0, "infer_s": 0.0},
    # {"approach": "Embeddings + LR", "accuracy": 0.00, "f1_macro": 0.00, "train_s": 0.0, "infer_s": 0.0},
    # {"approach": "DistilBERT FT", "accuracy": 0.00, "f1_macro": 0.00, "train_s": 0.0, "infer_s": None},
])
results



## 7. Notes & Tips

- **Imbalance:** If your dataset is imbalanced, consider:
  - Class weights (e.g., `class_weight='balanced'` in Logistic Regression; `scale_pos_weight` or per-class weights in XGBoost).
  - Stratified splits; macro-F1 is already more robust than accuracy.
  - Data augmentation (back-translation, synonym replacement) — test carefully.

- **Feature ablations:** Try TF‑IDF with (1,1) vs (1,2) n‑grams; prune `max_features`; adjust `min_df`/`max_df`.

- **Sentence-Transformer choices:** `all-MiniLM-L6-v2` is a great speed/quality tradeoff. Larger models can improve accuracy.

- **DistilBERT:** Start with 2–3 epochs; if GPU budget allows, try 4–5. Tune LR (1e-5 .. 5e-5), batch size, and warmup steps.

- **Export models:** 
  - XGBoost: `xgb.save_model("xgb_20ng.json")`
  - Logistic Regression: `joblib.dump(clf, "lr_emb.joblib")`
  - DistilBERT: `trainer.save_model("distilbert_20ng_model")`
